In [1]:
import os
import tifffile
import numpy as np
from pycromanager import Dataset

In [2]:
import numpy as np

def pbc_calculation(movie, N):
    """
    Correct signal decay across all frames to match the total intensity
    of each z-plane to that of the specified frame N, using NumPy array operations.
   
    Parameters:
    - movie: numpy array with dimensions (t, z, x, y, ch),
             where t = timepoints, z = z-planes, x and y = spatial dimensions, ch = channels.
    - N: int, the frame index used as the reference for intensity matching.
   
    Returns:
    - corrected_movie: numpy array with corrected signal intensities.
    """
    # Calculate the mean intensity for each z-plane and channel at each timepoint
    mean_intensities = movie.mean(axis=(2, 3))
   
    # Extract the mean intensity for the reference frame N for each z-plane and channel
    reference_intensities = mean_intensities[N]
   
    # Calculate the correction factors for each timepoint, z-plane, and channel
    correction_factors = np.ones_like(mean_intensities)  # Initialize to ones

    # Correct channel 0 using the reference intensity at N
    correction_factors[:, :, 0] = reference_intensities[:, 0] / mean_intensities[:, :, 0]

    # If there are more than 1 channel, correct channel 1 using the max mean value in the first 5 time points
    if movie.shape[-1] > 1:
        # Extract the mean intensity for the first 5 frames for each z-plane and channel 1
        mean_intensities_first_5 = mean_intensities[:5]
        # Find the maximum average value within the first 5 time points for each z-plane and channel 1
        max_mean_intensity_first_5_ch1 = mean_intensities_first_5[:, :, 1].max(axis=0)
        correction_factors[:, :, 1] = max_mean_intensity_first_5_ch1 / mean_intensities[:, :, 1]

    # If there are more than 2 channels, correct channel 2 using the reference intensity at N
    if movie.shape[-1] > 2:
        correction_factors[:, :, 2] = reference_intensities[:, 2] / mean_intensities[:, :, 2]
   
    # Avoid division by zero by setting any infinite correction factors to 1
    correction_factors[np.isinf(correction_factors)] = 1
   
    # Ensure correction_factors shape matches movie shape for broadcasting
    correction_factors_reshaped = correction_factors[:, :, np.newaxis, np.newaxis, :]
   
    # Apply the correction factors to the entire movie
    corrected_movie = movie * correction_factors_reshaped
   
    return corrected_movie

In [3]:
import numpy as np
import trackpy as tp
import pandas as pd

def best_z_plane(img, channel, z_stack_group_size):
    """
This function selects the best z-plane for each frame in a 5D image stack based on the total brightness of the top particles detected in each z-group.

Parameters:
img (numpy.ndarray): A 5D numpy array representing the image stack. The dimensions should be in the order (T, Z, Y, X, C), where T is time, Z is the z-stack, Y and X are spatial dimensions, and C is the channel.
channel (int): The index of the channel to process.
z_stack_group_size (int): The number of consecutive z-slices to group together for processing. The function will iterate over the z dimension in groups of this size.

Returns:
new_img (numpy.ndarray): A 5D numpy array representing the new image stack with the best z-plane selected for each frame.

How it works:
The function iterates over the time dimension and for each frame, it iterates over the z dimension in groups of z_stack_group_size. For each z-group, it combines the images into a single image using max projection along the z dimension. It then performs particle detection on the combined image and calculates the total brightness of the top 5-10 particles. The total brightness is multiplied by the square of the total number of particles to give more weight to the number of particles. The z-group with the highest score is selected as the best for the current frame. The function then selects the frames and z-stacks in the current groups and adds them to a list. Finally, it concatenates the new frames along the time dimension to create the new image.

Assumptions:
- The input image is a 5D numpy array with dimensions in the order (T, Z, Y, X, C).
- The channel index provided is valid for the input image.
- The z_stack_group_size is less than or equal to the size of the Z dimension of the input image.
- The particle detection function tp.locate can successfully detect particles in the images.
"""
    # Select the channel
    img_channel = img[:, :, channel, :, :]

    # Initialize the list to store the new frames
    new_frames = []

    # Iterate over time dimension
    for t in range(img_channel.shape[0]):
        # Initialize the best total brightness and best z-group for this frame
        best_total_brightness = 0
        best_z_group = None

        # Iterate over z dimension in groups of z_stack_group_size, starting from the second slice
        for z in range(1, img_channel.shape[1] - (z_stack_group_size - 1)):
            # Combine the images in the current z group into a single image
            # Use max projection along the z dimension
            img_group = np.max(img_channel[t, z:z + z_stack_group_size, :, :], axis=0)

            # Perform detection
            f = tp.locate(img_group, diameter=5, minmass = 700)
            f["mass"] = pd.to_numeric(f["mass"])
            # Calculate the total brightness of the top 5-10 particles
            top_particles = f.nlargest(min(10, len(f)), 'mass')
            group_brightness = top_particles['mass'].sum()

            # Multiply the total brightness by the square of the total number of particles to give more weight to the number of particles
            group_score = group_brightness * len(f) ** 2

            # If this group has a higher score than the current best, update the best
            if group_score > best_total_brightness:
                best_total_brightness = group_score
                best_z_group = z

        # Select the frames and z-stacks in the current groups
        frames = img[t, best_z_group:best_z_group + z_stack_group_size, :, :, :]

        # Add the frames to the list
        new_frames.append(frames[np.newaxis, ...])

    # Concatenate the new frames along the time dimension
    new_img = np.concatenate(new_frames, axis=0).astype(np.uint16)

    return new_img

In [6]:
#This code will take the terminator data file and the image file and convert the positions into each a separate .tif file.
#The code will also correct the signal decay across all channels to match the total intensity of each channel to that of the specified channel N.
#The code will also select the best z-plane for each frame in a 5D image stack based on the total brightness of the top particles detected in each z-group.
#This is most useful to split images by position
# Define the path to your dataset
data_path = r"Z:\SharedExternal\Daugherty\Imaging\PARP13S\20260528\65ms_20t_0I\Dish01\20260528_2_1"

# Ask the user for their preference
print("What do you want to save?")
print("1. Just the PBC image")
print("2. Just the original image")
print("3. Both")
user_choice = input("Enter your choice (1, 2, or 3): ")


# Create a dataset object
dataset = Dataset(data_path)

# Get the number of positions in the dataset
num_positions = len(set(info['position'] for info in dataset.get_index_keys()))

# Get the number of time points in the dataset
num_time_points = len(set(info['time'] for info in dataset.get_index_keys()))

# Get the number of z stacks in the dataset
num_z_stacks = len(set(info['z'] for info in dataset.get_index_keys()))

# Get the number of channels in the dataset
num_channels = len(dataset.get_channel_names())

# Iterate over positions
for position in range(num_positions):
    print(f"Processing Position {position + 1}...")
    
    # Create an empty hyperstack array with the correct shape
    hyperstack_data = np.empty((num_time_points, num_z_stacks, num_channels, 512, 512), dtype=np.uint16)

    # Rearrange the data to the hyperstack format
    for t in range(num_time_points):
        for z in range(num_z_stacks):
            for c in range(num_channels):
                # Use the read_image method to retrieve the image data for a specific frame
                img = dataset.read_image(channel=c, z=z, time=t, position=position)
                hyperstack_data[t, z, c] = img
                
    # Ensure that hyperstack_data is in the correct order (t, z, x, y, ch)
    hyperstack_data_t = np.transpose(hyperstack_data, (0, 1, 3, 4, 2))
         
    pcb_img = pbc_calculation(hyperstack_data_t, 0)
    pbc_img_t = np.transpose(pcb_img, (0, 1, 4, 2, 3))
    best_z_plane_img = best_z_plane(pbc_img_t, 0, 7)
    
        # Save the images based on the user's choice
    if user_choice == '1':
        output_path = os.path.join(data_path, f"Cell0{position + 1}_pbc.tif")
        tifffile.imsave(output_path, best_z_plane_img.astype(np.uint16), imagej=True)
    elif user_choice == '2':
        output_path = os.path.join(data_path, f"Cell0{position + 1}.tif")
        tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)
    elif user_choice == '3':
        output_path = os.path.join(data_path, f"Cell0{position + 1}_pbc.tif")
        tifffile.imsave(output_path, best_z_plane_img.astype(np.uint16), imagej=True)
        output_path = os.path.join(data_path, f"Cell0{position + 1}.tif")
        tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)
    else:
        print("Invalid choice. Please enter 1, 2, or 3.")
    


What do you want to save?
1. Just the PBC image
2. Just the original image
3. Both
Dataset opened                 
Processing Position 1...


C:\Users\AdamC\AppData\Local\Temp\ipykernel_30156\3543036435.py:59: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)


Processing Position 2...


C:\Users\AdamC\AppData\Local\Temp\ipykernel_30156\3543036435.py:59: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)


Processing Position 3...


...

C:\Users\AdamC\AppData\Local\Temp\ipykernel_30156\3543036435.py:59: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)


Processing Position 12...


C:\Users\AdamC\AppData\Local\Temp\ipykernel_30156\3543036435.py:59: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)


Processing Position 13...


C:\Users\AdamC\AppData\Local\Temp\ipykernel_30156\3543036435.py:59: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_path, hyperstack_data.astype(np.uint16), imagej=True)


In [4]:
#This code will take the terminator data file and the image file and convert the positions into each a separate .tif file.
#This is most useful to split images by position
# Define the path to your dataset
data_path = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\BoxB_MS2\20250820\65msExp_20T_0I\Dish01\20250820_2_1"

# Create a dataset object
dataset = Dataset(data_path)

# Get the number of positions in the dataset
num_positions = len(set(info['position'] for info in dataset.get_index_keys()))

# Get the number of time points in the dataset
num_time_points = len(set(info['time'] for info in dataset.get_index_keys()))

# Get the number of z stacks in the dataset
num_z_stacks = len(set(info['z'] for info in dataset.get_index_keys()))

# Get the number of channels in the dataset
num_channels = len(dataset.get_channel_names())

# Iterate over positions
for position in range(num_positions):
    print(f"Processing Position {position + 1}...")
    
    # Create an empty hyperstack array with the correct shape
    hyperstack_data = np.empty((num_time_points, num_z_stacks, num_channels, 512, 512), dtype=np.uint16)

    # Rearrange the data to the hyperstack format
    for t in range(num_time_points):
        for z in range(num_z_stacks):
            for c in range(num_channels):
                # Use the read_image method to retrieve the image data for a specific frame
                img = dataset.read_image(channel=c, z=z, time=t, position=position)
                hyperstack_data[t, z, c] = img
    
    # Save the hyperstack in the same directory with the position number in the filename
    output_path = os.path.join(data_path, f"Cell0{position + 1}.tif")
    tifffile.imsave(output_path, hyperstack_data, imagej=True)
    print(f"Hyperstack saved to: {output_path}")


NameError: name 'Dataset' is not defined